In [3]:
import sys
import unittest
%run Modelo.ipynb
%run Servicios.ipynb

sys.modules["modelo"] = sys.modules["__main__"]
sys.modules["Modelo"] = sys.modules["__main__"]
sys.modules["Servicios"] = sys.modules["__main__"]
%run Controlador.ipynb
class VistaFalsa:
    def __init__(self, textos=None, elegir=None, enteros=None, si_no=None):
        self._textos = iter(textos or [])
        self._elegir = elegir
        self._enteros = iter(enteros or [])
        self._si_no = iter(si_no or [])
        self.mensajes = []
        self.errores = []

    def limpiar_pantalla(self): pass
    def esperar_enter(self, mensaje): pass
    def mostrar_rol_privado(self, jugador, categoria): pass
    def mostrar_pistas(self, pistas): pass
    def esperar_debate(self, segundos): pass
    def mostrar_resultado(self, resultado): pass

    def mostrar_mensaje(self, texto):
        self.mensajes.append(texto)

    def mostrar_error(self, texto):
        self.errores.append(texto)

    def pedir_texto(self, mensaje):
        return next(self._textos)

    def elegir_opcion(self, mensaje, opciones):
        if callable(self._elegir):
            return self._elegir(mensaje, opciones)
        return next(self._elegir)

    def pedir_entero(self, mensaje, minimo, maximo):
        return next(self._enteros)

    def pedir_si_no(self, mensaje):
        return next(self._si_no)


def crear_partida(nombres):
    palabra = PalabraSecreta("Playa", Categoria("Lugares"))
    jugadores = [Jugador(i, n) for i, n in enumerate(nombres, start=1)]
    configuracion = ConfiguracionPartida(3, 60, "Lugares", True, 20)
    partida = Partida(
        jugadores,
        palabra,
        configuracion,
        AsignadorRolesAleatorio(),
        ContadorVotosMayoria(),
        [ValidadorPalabraProhibida(palabra), ValidadorLongitud(20)],
    )
    partida.iniciar()
    return partida, jugadores, configuracion
class TestControladorVotacion(unittest.TestCase):
    def preparar(self, elegir):
        partida, jugadores, _ = crear_partida(["Danna", "Daniela", "Emmanuel", "Leonardo"])
        partida.avanzar_fase()   # PISTAS -> DEBATE
        partida.avanzar_fase()   # DEBATE -> VOTACION
        vista = VistaFalsa(elegir=elegir)
        return ControladorVotacion(vista, partida), vista, jugadores

    def test_elimina_al_mas_votado(self):
        def elegir(mensaje, opciones):
            votante = mensaje.split(",")[0]
            return {"Danna": 0, "Daniela": 0, "Emmanuel": 1, "Leonardo": 1}[votante]

        controlador, vista, jugadores = self.preparar(elegir)
        acusado = controlador.ejecutar()

        self.assertIs(acusado, jugadores[1])       # Daniela (3 votos)
        self.assertTrue(jugadores[1].eliminado)
        self.assertTrue(any("Daniela fue el más votado" in m for m in vista.mensajes))

    def test_empate_no_elimina_a_nadie(self):
        # Danna vota 0 (Daniela), Daniela vota 0 (Danna), Emmanuel vota 1 (Daniela), Leonardo vota 0 (Danna) -> 2 a 2
        def elegir(mensaje, opciones):
            votante = mensaje.split(",")[0]
            return {"Danna": 0, "Daniela": 0, "Emmanuel": 1, "Leonardo": 0}[votante]

        controlador, vista, jugadores = self.preparar(elegir)
        acusado = controlador.ejecutar()

        self.assertIsNone(acusado)
        self.assertTrue(all(not j.eliminado for j in jugadores))
        self.assertTrue(any("empate" in m.lower() for m in vista.mensajes))

class TestControladorRonda(unittest.TestCase):
    def test_repite_pista_invalida_y_completa_la_ronda(self):
        partida, jugadores, configuracion = crear_partida(["Danna", "Daniela", "Emmanuel", "Leonardo"])

        # Todos votan por el índice 0:
        # Danna vota a Daniela (0); Daniela, Emmanuel y Leonardo votan a Danna (0) -> 3 votos para Danna
        def elegir(mensaje, opciones):
            return 0

        # "la playa" falla por palabra prohibida; luego entran 4 pistas válidas para los 4 jugadores
        vista = VistaFalsa(
            textos=["la playa", "arena", "sol", "mar", "brisa"],
            elegir=elegir,
        )
        ronda = ControladorRonda(vista, partida, ControladorVotacion(vista, partida), configuracion)
        eliminado = ronda.ejecutar()

        self.assertEqual(len(vista.errores), 1)
        self.assertEqual(
            [p.get_texto() for p in partida.ronda_actual.pistas],
            ["arena", "sol", "mar", "brisa"],
        )
        self.assertEqual(partida.ronda_actual.fase, FaseJuego.RESULTADO)
        self.assertIs(eliminado, jugadores[0])     # Danna eliminada
class TestPartidaYModelo(unittest.TestCase):
    def test_voto_a_si_mismo_lanza_excepcion(self):
        j1 = Jugador(1, "Danna")
        with self.assertRaises(VotoInvalidoException):
            j1.votar(j1)

    def test_civiles_ganan_si_eliminan_impostor_y_falla_adivinanza(self):
        partida, jugadores, _ = crear_partida(["Danna", "Daniela", "Emmanuel", "Leonardo"])
        impostor = next(j for j in jugadores if isinstance(j.get_rol(), Impostor))
        impostor.eliminar()

        resultado = partida.determinar_ganador(intento_impostor="Montaña")
        self.assertEqual(resultado.ganador, Ganador.CIVILES)

    def test_impostor_gana_si_adivina_palabra_en_ultima_oportunidad(self):
        partida, jugadores, _ = crear_partida(["Danna", "Daniela", "Emmanuel", "Leonardo"])
        impostor = next(j for j in jugadores if isinstance(j.get_rol(), Impostor))
        impostor.eliminar()

        resultado = partida.determinar_ganador(intento_impostor="  playa ")
        self.assertEqual(resultado.ganador, Ganador.IMPOSTOR)

if __name__ == "__main__":
    loader = unittest.TestLoader()
    suite = unittest.TestSuite([
        loader.loadTestsFromTestCase(TestControladorVotacion),
        loader.loadTestsFromTestCase(TestControladorRonda),
        loader.loadTestsFromTestCase(TestPartidaYModelo),
    ])
    runner = unittest.TextTestRunner(verbosity=2)
    runner.run(suite)

.................
----------------------------------------------------------------------
Ran 17 tests in 0.012s

OK
test_elimina_al_mas_votado (__main__.TestControladorVotacion.test_elimina_al_mas_votado) ... ok
test_empate_no_elimina_a_nadie (__main__.TestControladorVotacion.test_empate_no_elimina_a_nadie) ... ok
test_repite_pista_invalida_y_completa_la_ronda (__main__.TestControladorRonda.test_repite_pista_invalida_y_completa_la_ronda) ... ok
test_civiles_ganan_si_eliminan_impostor_y_falla_adivinanza (__main__.TestPartidaYModelo.test_civiles_ganan_si_eliminan_impostor_y_falla_adivinanza) ... ok
test_impostor_gana_si_adivina_palabra_en_ultima_oportunidad (__main__.TestPartidaYModelo.test_impostor_gana_si_adivina_palabra_en_ultima_oportunidad) ... ok
test_voto_a_si_mismo_lanza_excepcion (__main__.TestPartidaYModelo.test_voto_a_si_mismo_lanza_excepcion) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.009s

OK
